# 07l — Historical imagery unblinding synthesis

This stage unblinds the completed 9-candidate Stage-07k visual review and joins it with Stage-07j behavioral persistence plus BCL-primary context.

It creates **no composite OFFICE score** and performs **no automatic promotion**. The goal is to see whether the imagery-defined contexts line up with the existing behavioral evidence once the blind is removed.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))

def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)
if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)

spec=importlib.util.spec_from_file_location('s07l',REPO_DIR/'analysis'/'07l_imagery_unblinding_synthesis.py')
assert spec is not None and spec.loader is not None
s07l=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07l
spec.loader.exec_module(s07l)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print(s07l.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07l_imagery_unblinding_synthesis'; C.mkdir(parents=True,exist_ok=True)
C07K=V/'cache'/'cp2_v2'/'07k_historical_imagery_review'
C07J=V/'cache'/'cp2_v2'/'07j_near_miss_office_audit'

def find_review_csv():
    env=os.environ.get('GEOLIFE_07K_REVIEW_CSV')
    candidates=[]
    if env: candidates.append(Path(env))
    candidates.extend([
        Path('/mnt/data/near_miss_9_blinded_review_manifest_filled.csv'),
        C07K/'near_miss_9_blinded_review_manifest_filled.csv',
        C07K/'near_miss_9_blinded_review_manifest_private.csv',
    ])
    for p in candidates:
        if p.exists(): return p
    matches=sorted(V.glob('**/near_miss_9_blinded_review_manifest_filled.csv'))
    if matches: return matches[0]
    raise FileNotFoundError(
        'Upload/copy near_miss_9_blinded_review_manifest_filled.csv into /mnt/data, '
        'or set GEOLIFE_07K_REVIEW_CSV to its path.'
    )

REVIEW=find_review_csv()
KEY=C07K/'historical_imagery_unblinding_key_private.pkl'
AUDIT=C07J/'near_miss_office_audit_private.pkl'
for p in [KEY,AUDIT]:
    if not p.exists(): raise FileNotFoundError(f'Missing upstream artifact: {p}')

review=pd.read_csv(REVIEW)
key=pd.read_pickle(KEY)
audit=pd.read_pickle(AUDIT)
print('review file:',REVIEW)
print('review rows:',len(review),'key rows:',len(key),'audit rows:',len(audit))


In [ ]:
panel=s07l.build_unblinded_panel(review,key,audit)
assert len(panel)==9
assert set(panel['audit_group'])=={'margin_near','share_near'}
assert panel['visual_context_class'].notna().all()
print('unblinding validation: PASS (9/9 near-miss candidates matched exactly)')

visual_summary=s07l.summarize_visual_context(panel)
behavior_visual=s07l.summarize_behavior_by_visual_bucket(panel)
signatures=s07l.summarize_cross_source_signatures(panel)
snapshot=s07l.policy_snapshot(panel)
case_matrix=s07l.private_case_matrix(panel)


In [ ]:
print('VISUAL CONTEXT BY NEAR-MISS FAMILY')
display(visual_summary)

print('BEHAVIOR × VISUAL BUCKET')
display(behavior_visual)

print('CROSS-SOURCE SIGNATURES')
display(signatures)

print('POLICY SNAPSHOT')
display(snapshot)


In [ ]:
# Private case-level matrix: contains user IDs after unblinding.
panel.to_pickle(C/'imagery_unblinded_panel_private.pkl')
case_matrix.to_pickle(C/'imagery_behavior_bcl_case_matrix_private.pkl')

# Aggregate outputs only.
visual_summary.to_csv(C/'visual_context_by_near_miss_family.csv',index=False)
behavior_visual.to_csv(C/'behavior_by_visual_bucket.csv',index=False)
signatures.to_csv(C/'cross_source_signature_summary.csv',index=False)
snapshot.to_csv(C/'policy_snapshot.csv',index=False)
print('saved:',C)


## Interpretation boundary

- `office_like` means only the imagery class `large_office_commercial_like_complex`.
- `institutional_or_daytime_compatible` includes education, healthcare/institutional, and industrial morphology. These may explain a recurring daytime anchor but do **not** establish OFFICE.
- `office_contradictory_context` includes residential, transport, construction/vacant, and recreation morphology.
- `indeterminate` contains mixed urban, other visible structure, and ambiguous imagery.
- Same-year imagery proxies are approximate historical context; they are not exact acquisition-date proof.
- No row is promoted automatically. Any production change must be based on group-level robustness, not a single visually appealing case.
